# Train &amp; Evaluate the Formation Classifier

LightGBM multiclass classifier predicting `P(SHOTGUN), P(UNDER CENTER), P(PISTOL)` given pre-snap game state, coach tendencies, and the already-decided `play_type` (this model sits downstream of `play_type` in the pipeline: play_type → **formation** → personnel → result). See `src/fb_models/modeling/plays.py` (shared feature engineering) and `src/fb_models/modeling/formation/` (this model's feature selection + train/predict).

**Label note**: nflverse changed its `offense_formation` tracking methodology starting in 2023 — 2019-2022 break "under center" into `SINGLEBACK`/`I_FORM`/`JUMBO`/`WILDCAT` and separately track `EMPTY`, while 2023+ collapses the true under-center variants into a single `UNDER CENTER` and drops `EMPTY` entirely. `modeling/plays.py`'s `_FORMATION_REMAP` normalizes all of this to the 3-class scheme that matches nflverse's current (and presumably future) convention: `EMPTY` (confirmed 98.5% shotgun snaps on real data) → `SHOTGUN`; the rest → `UNDER CENTER`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, log_loss

from fb_models.dataset import load_pbp_dataset, load_participation_dataset, load_games_dataset
from fb_models.modeling.plays import build_plays
from fb_models.modeling.formation.features import FORMATION_FEATURE_COLS, select_formation_features
from fb_models.modeling.formation.train import train_formation_classifier
from fb_models.modeling.formation.predict import build_live_feature_row, predict_formation_probs

In [2]:
# Season-based split, matching play_type's convention: train on everything
# before TEST_SEASON, hold out TEST_SEASON entirely. Loading the full range
# up front matters here -- the expanding coach-tendency and previous-play
# features need continuous history leading into the test season.
SEASONS = list(range(2019, 2025))
TEST_SEASON = 2024

In [3]:
pbp = load_pbp_dataset(seasons=SEASONS)
participation = load_participation_dataset(seasons=SEASONS)
games = load_games_dataset()

plays = build_plays(pbp, participation, games)
plays.shape

(221042, 89)

In [4]:
train_plays = plays[plays["season"] < TEST_SEASON]
test_plays = plays[plays["season"] == TEST_SEASON]

train_df = select_formation_features(train_plays)
test_df = select_formation_features(test_plays)

print(f"train: {len(train_df):,} plays (seasons {SEASONS[0]}-{TEST_SEASON - 1})")
print(f"test:  {len(test_df):,} plays (season {TEST_SEASON})")
print()
print("label distribution (train):")
print(train_df["offense_formation"].value_counts(normalize=True))

train: 165,444 plays (seasons 2019-2023)
test:  33,310 plays (season 2024)

label distribution (train):
offense_formation
SHOTGUN         0.638494
UNDER CENTER    0.326310
PISTOL          0.035196
Name: proportion, dtype: float64


In [5]:
clf = train_formation_classifier(train_df)
clf.classes_

array(['PISTOL', 'SHOTGUN', 'UNDER CENTER'], dtype=object)

## Evaluation on the held-out season

In [6]:
X_test = test_df[FORMATION_FEATURE_COLS]
y_test = test_df["offense_formation"]

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)

accuracy = (y_pred == y_test.to_numpy()).mean()
loss = log_loss(y_test, y_proba, labels=clf.classes_)

print(f"accuracy: {accuracy:.3f}")
print(f"log_loss: {loss:.3f}")
print()
print(classification_report(y_test, y_pred, labels=clf.classes_, zero_division=0))

accuracy: 0.720
log_loss: 0.660

              precision    recall  f1-score   support

      PISTOL       0.48      0.13      0.21      3115
     SHOTGUN       0.76      0.89      0.82     20561
UNDER CENTER       0.63      0.55      0.59      9634

    accuracy                           0.72     33310
   macro avg       0.62      0.52      0.54     33310
weighted avg       0.70      0.72      0.70     33310



In [7]:
labels = list(clf.classes_)
cm = confusion_matrix(y_test, y_pred, labels=labels)
pd.DataFrame(cm, index=labels, columns=labels)

,PISTOL,SHOTGUN,UNDER CENTER
PISTOL,412,1714,989
SHOTGUN,143,18274,2144
UNDER CENTER,309,4020,5305


### Calibration sanity check

Same rationale as play_type: the simulation samples from the predicted distribution rather than taking the argmax, so mean predicted probabilities should track real-world marginal rates.

In [8]:
mean_predicted_probs = pd.Series(y_proba.mean(axis=0), index=labels, name="mean_predicted_prob")
actual_rates = y_test.value_counts(normalize=True).reindex(labels).rename("actual_rate")

pd.concat([mean_predicted_probs, actual_rates], axis=1)

,mean_predicted_prob,actual_rate
PISTOL,0.064408,0.093515
SHOTGUN,0.663960,0.617262
UNDER CENTER,0.271631,0.289222


## Example predictions

`predict_formation_probs` needs `play_type` as an input (the pipeline order is play_type → formation) plus a coach-tendency snapshot, same shape as play_type's `build_live_feature_row`.

In [9]:
situational_cols = ["down", "ydstogo", "yardline_100", "goal_to_go", "score_differential", "play_type"]

scenarios = {
    "1st & 10, midfield, tied, run": [1, 10, 50, 0, 0, "run"],
    "3rd & long, down 3, pass": [3, 12, 60, 0, -3, "pass"],
    "Goal to go, 2nd down, run": [2, 3, 3, 1, 3, "run"],
    "4th & short, field goal range, pass": [4, 2, 25, 0, -2, "pass"],
}

rows = []
for name, values in scenarios.items():
    sample_play = test_df.iloc[[0]].copy()
    sample_play.loc[:, situational_cols] = values
    sample_play["play_type"] = sample_play["play_type"].astype("category")

    probs = predict_formation_probs(clf, sample_play)
    rows.append({"scenario": name, **{f"prob_{k.lower().replace(' ', '_')}": v for k, v in probs.items()}})

sample_plays = pd.DataFrame(rows).set_index("scenario")
sample_plays

,prob_pistol,prob_shotgun,prob_under_center
scenario,,,
"1st & 10, midfield, tied, run",0.170778,0.147883,0.681339
"3rd & long, down 3, pass",0.020585,0.965039,0.014376
"Goal to go, 2nd down, run",0.112305,0.211648,0.676047
"4th & short, field goal range, pass",0.058139,0.812514,0.129347


## Save artifact

Retrains on the full dataset (no held-out season) before saving.

In [10]:
import joblib

from fb_models.config import MODELS_DIR

full_df = select_formation_features(plays)
final_clf = train_formation_classifier(full_df)

MODELS_DIR.mkdir(exist_ok=True)

artifact_path = MODELS_DIR / "formation_classifier.joblib"
joblib.dump(final_clf, artifact_path)
print(f"saved {artifact_path} ({len(full_df):,} plays, seasons {SEASONS[0]}-{SEASONS[-1]})")

saved /home/davis/projects/fb_models/models/formation_classifier.joblib (198,754 plays, seasons 2019-2024)
